In [1]:
import json
import re
import random
from collections import Counter
from pathlib import Path

import spacy
from spacy.tokens import DocBin

In [2]:
from huggingface_hub import hf_hub_download
INPUT_PATH = Path(hf_hub_download(
    repo_id="alrosait/pii-synthetic-ru",
    filename="synthetic_pii.jsonl",
    repo_type="dataset",
))
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_RATIO = 0.9
RANDOM_SEED = 42

In [3]:
records = []
with open(INPUT_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Загружено записей: {len(records)}")

entity_type_counts = Counter(r["entity_type"] for r in records)
print("По типу entity_type:", dict(entity_type_counts))

Загружено записей: 4500
По типу entity_type: {'BOTH': 1409, 'NEGATIVE': 1026, 'ADDRESS_ONLY': 952, 'NAME_ONLY': 929, 'NAME_CONTEXT': 184}


In [4]:
nlp = spacy.blank("ru")


def record_to_doc(record: dict):
    """Конвертирует одну запись JSONL в spaCy Doc с аннотациями NER."""
    text = record["text"]
    entities = record.get("entities", [])

    doc = nlp.make_doc(text)

    spans = []
    for ent in entities:
        ent_text = ent["text"]
        label = ent["type"]  # NAME или ADDRESS

        # Находим все вхождения текста сущности в тексте примера
        for match in re.finditer(re.escape(ent_text), text):
            span = doc.char_span(
                match.start(), match.end(),
                label=label,
                alignment_mode="expand"
            )
            if span is not None:
                spans.append(span)

    doc.ents = spacy.util.filter_spans(spans)
    return doc

In [5]:
docs = []
skipped = 0

for record in records:
    try:
        doc = record_to_doc(record)
        docs.append(doc)
    except Exception as e:
        skipped += 1

print(f"Конвертировано документов: {len(docs)}")
print(f"Пропущено из-за ошибок: {skipped}")

ent_counts = Counter(ent.label_ for doc in docs for ent in doc.ents)
print("Сущностей всего:", dict(ent_counts))

neg_docs = sum(1 for doc in docs if len(doc.ents) == 0)
print(f"Негативных примеров (без сущностей): {neg_docs}")

Конвертировано документов: 4500
Пропущено из-за ошибок: 0
Сущностей всего: {'ADDRESS': 2660, 'NAME': 2941}
Негативных примеров (без сущностей): 1026


In [6]:
random.seed(RANDOM_SEED)
random.shuffle(docs)

split_idx = int(len(docs) * TRAIN_RATIO)
train_docs = docs[:split_idx]
dev_docs = docs[split_idx:]

print(f"Train: {len(train_docs)} docs")
print(f"Dev:   {len(dev_docs)} docs")

Train: 4050 docs
Dev:   450 docs


In [7]:
def save_docbin(docs, path):
    db = DocBin()
    for doc in docs:
        db.add(doc)
    db.to_disk(path)
    print(f"Сохранено {len(docs)} docs → {path}")


save_docbin(train_docs, OUTPUT_DIR / "train_synth.spacy")
save_docbin(dev_docs, OUTPUT_DIR / "dev_synth.spacy")

Сохранено 4050 docs → ..\data\train_synth_2.spacy
Сохранено 450 docs → ..\data\dev_synth_2.spacy


In [8]:
print("=== Train ===")
train_ents = Counter(ent.label_ for doc in train_docs for ent in doc.ents)
train_neg = sum(1 for doc in train_docs if len(doc.ents) == 0)
print(f"  Документов: {len(train_docs)}")
print(f"  Сущностей: {dict(train_ents)}")
print(f"  Негативных примеров: {train_neg}")

print("=== Dev ===")
dev_ents = Counter(ent.label_ for doc in dev_docs for ent in doc.ents)
dev_neg = sum(1 for doc in dev_docs if len(doc.ents) == 0)
print(f"  Документов: {len(dev_docs)}")
print(f"  Сущностей: {dict(dev_ents)}")
print(f"  Негативных примеров: {dev_neg}")

=== Train ===
  Документов: 4050
  Сущностей: {'NAME': 2640, 'ADDRESS': 2396}
  Негативных примеров: 927
=== Dev ===
  Документов: 450
  Сущностей: {'NAME': 301, 'ADDRESS': 264}
  Негативных примеров: 99


In [9]:
# Проверка: читаем сохранённые файлы обратно
train_check = list(DocBin().from_disk(OUTPUT_DIR / "train_synth.spacy").get_docs(nlp.vocab))
dev_check = list(DocBin().from_disk(OUTPUT_DIR / "dev_synth.spacy").get_docs(nlp.vocab))

assert len(train_check) == len(train_docs)
assert len(dev_check) == len(dev_docs)

print("Проверка пройдена.")
print(f"train_synth.spacy: {len(train_check)} docs")
print(f"dev_synth.spacy:   {len(dev_check)} docs")

# Пример нескольких аннотированных документов
print("\nПримеры из train:")
for doc in train_check[:3]:
    print(f"  Текст: {doc.text[:80]}")
    for ent in doc.ents:
        print(f"    [{ent.label_}] {ent.text!r}")

Проверка пройдена.
train_synth_2.spacy: 4050 docs
dev_synth_2.spacy:   450 docs

Примеры из train:
  Текст: Л.Р.Евдокимова обратилась в отделение в связи с утратой карты. К сожалению, доку
    [NAME] 'Л.Р.Евдокимова'
    [ADDRESS] 'Уральская 198'
    [ADDRESS] 'Герцена 98'
  Текст: Обратился в МФЦ по вопросу замены документов. Сотрудник попросил уточнить распол
  Текст: В связи с плановым техническим обслуживанием автомобилей сервисный центр перенос
